# The OOM boundary: baseline vs Stream-CQSA as N grows

This notebook sets the sequence length **N explicitly** and runs the monolithic baseline
(PyTorch SDPA, FlashAttention-2 backend) and Stream-CQSA on the same inputs at each N.
A **memory cap** simulates a small device so the boundary arrives in seconds instead of hours:

* below the boundary both run; Stream-CQSA costs a little more time (it does 9/7 of the pair work
  and streams from host memory) and is, if anything, slightly *more* accurate against float64;
* past the boundary the baseline raises `OutOfMemoryError` and Stream-CQSA keeps going, exact.

The planner picks the configuration (monolithic call while it fits, then the decomposition depth,
quorum set and accumulator placement) from the capped budget automatically.

In [1]:
import os, gc, time, torch, torch.nn.functional as F
os.environ.setdefault("CQSA_CUDA_MODULE", "cqsa_cuda_next_v11"); os.environ.setdefault("CQSA_CUDA_MODULE_NONCAUSAL", "cqsa_cuda_next_v9")
from stream_cqsa.autoconfig import auto_attention, hardware_from_dict, detect_hardware
from stream_cqsa.devkit import reference_rows, sample_rows, accuracy_vs_fp64
import stream_cqsa.interface as I
dev = torch.device("cuda"); B, H, D = 1, 8, 64
print(torch.cuda.get_device_name(0), "| kernel:", "Triton (no CUDA extension)" if I.cqsa_cuda is None else os.path.basename(I.cqsa_cuda.__file__))
CAP_GIB = 3.0
total = torch.cuda.get_device_properties(0).total_memory / 2**30
torch.cuda.set_per_process_memory_fraction(CAP_GIB / total)
hw = hardware_from_dict({"cuda:0": f"{CAP_GIB}GiB", "host": "200GiB"})
print(f"memory cap {CAP_GIB} GiB of {total:.0f} GiB  ->  the planner sees:", hw.summary())
def run(fn):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    try:
        out = fn(); torch.cuda.synchronize()
        return out, time.perf_counter() - t0, torch.cuda.max_memory_allocated() / 2**30, None
    except torch.cuda.OutOfMemoryError as e:
        gc.collect(); torch.cuda.empty_cache()
        return None, time.perf_counter() - t0, torch.cuda.max_memory_allocated() / 2**30, "OOM"

NVIDIA A100-SXM4-80GB | kernel: cqsa_cuda_next_v11.cpython-311-x86_64-linux-gnu.so
memory cap 3.0 GiB of 79 GiB  ->  the planner sees: devices[cuda:0=same 3.0/79.3 GiB] host 200 GiB, 8 cpus, link 25 GB/s


/home/yb2807/.conda/envs/CQS_ENV_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Sweep N

In [2]:
Ns = [16_384, 65_536, 131_072, 262_144, 524_288, 1_048_576]
rows = []
print(f"{'N':>9} | {'SDPA/FA-2':>22} | {'Stream-CQSA':>50} | {'err vs fp64':>19}")
for N in Ns:
    g = torch.Generator().manual_seed(0)
    q, k, v = (torch.randn(B, H, N, D, generator=g, dtype=torch.float32).to(torch.float16) for _ in range(3))   # host-resident
    rows_idx = sample_rows(N, 64); ref = reference_rows(q, k, v, rows_idx, causal=True, scale=D**-0.5)
    # baseline: inputs must be on the device
    def baseline():
        qd, kd, vd = (t.to(dev) for t in (q, k, v)); return F.scaled_dot_product_attention(qd, kd, vd, is_causal=True)
    ob, tb, mb, eb = run(baseline)
    # Stream-CQSA: planner on the capped budget, inputs streamed from the host when it decides so
    plan = {}
    def cqsa():
        out, p = auto_attention(q, k, v, causal=True, hardware=hw, allow_escalation=True); plan["p"] = p; return out
    oc, tc, mc, ec = run(cqsa)
    errb = accuracy_vs_fp64(ob, q, k, v, causal=True, scale=D**-0.5, rows=rows_idx, ref_rows=ref)["rel_fro"] if ob is not None else float("nan")
    errc = accuracy_vs_fp64(oc, q, k, v, causal=True, scale=D**-0.5, rows=rows_idx, ref_rows=ref)["rel_fro"] if oc is not None else float("nan")
    sb = f"OOM after {tb:.1f}s" if eb else f"{tb:6.2f} s, {mb:4.2f} GiB"
    sc = (f"OOM" if ec else f"{tc:6.2f} s, {mc:4.2f} GiB") + f"  [{plan['p'].name() if 'p' in plan else '-'}]"
    print(f"{N:>9} | {sb:>22} | {sc:>50} | {errb:8.1e} / {errc:8.1e}")
    rows.append(dict(N=N, baseline_s=tb, baseline_gib=mb, baseline=eb or "ok", cqsa_s=tc, cqsa_gib=mc, cqsa=ec or "ok", plan=plan["p"].name() if "p" in plan else None, err_baseline=errb, err_cqsa=errc))
    del q, k, v, ob, oc; gc.collect(); torch.cuda.empty_cache()

        N |              SDPA/FA-2 |                                        Stream-CQSA |         err vs fp64


    16384 |       0.17 s, 0.07 GiB |                     0.03 s, 0.09 GiB  [monolithic] |  2.8e-04 /  2.8e-04


    65536 |       0.04 s, 0.26 GiB |                     0.04 s, 0.32 GiB  [monolithic] |  2.9e-04 /  2.9e-04


   131072 |       0.13 s, 0.51 GiB |                     0.12 s, 0.64 GiB  [monolithic] |  3.0e-04 /  3.0e-04


   262144 |       0.43 s, 1.02 GiB |                     0.43 s, 1.27 GiB  [monolithic] |  3.2e-04 /  3.2e-04


   524288 |       1.62 s, 2.02 GiB |                     1.64 s, 2.52 GiB  [monolithic] |  4.7e-04 /  4.7e-04


  1048576 |         OOM after 0.4s |  20.95 s, 2.01 GiB  [cqsa c=13 itr=2 acc=cpu n_par=1 host-resident Q/K/V] |      nan /  2.0e-04


## Reading the table

* **Small N** (16K–512K under this cap): both run. The planner sees that the monolithic call fits the
  capped budget and uses it, so Stream-CQSA's time, memory and error are the baseline's (the ~0.25 GiB
  extra peak is the fp32 output it returns). Forced to decompose below the boundary (`itr=1`), it costs
  1.3–1.6x the monolithic time on this hardware and is slightly closer to float64 (fp32 merge).
* **Large N** (1M, past the cap): SDPA raises `OutOfMemoryError`; the planner picks `c=13, itr=2`,
  host-resident Q/K/V and a host accumulator, and Stream-CQSA returns the exact result (2.0e-4 vs
  float64) at a 2.0 GiB device peak — below the cap.

On an un-capped 80 GB A100 the same crossover happens at ~8M tokens for the forward and ~4M for the
backward (paper, Table 2); the memory cap only moves it to where a notebook can afford to run.

In [3]:
import json; print(json.dumps(rows, indent=1))

[
 {
  "N": 16384,
  "baseline_s": 0.16683178301900625,
  "baseline_gib": 0.0711679458618164,
  "baseline": "ok",
  "cqsa_s": 0.0269344300031662,
  "cqsa_gib": 0.0867924690246582,
  "cqsa": "ok",
  "plan": "monolithic",
  "err_baseline": 0.00027896280098680114,
  "err_cqsa": 0.00027884870993609347
 },
 {
  "N": 65536,
  "baseline_s": 0.042309935903176665,
  "baseline_gib": 0.2601327896118164,
  "baseline": "ok",
  "cqsa_s": 0.041742388042621315,
  "cqsa_gib": 0.3226323127746582,
  "cqsa": "ok",
  "plan": "monolithic",
  "err_baseline": 0.0002877243107716292,
  "err_cqsa": 0.000287660240070148
 },
 {
  "N": 131072,
  "baseline_s": 0.1305667629931122,
  "baseline_gib": 0.5120859146118164,
  "baseline": "ok",
  "cqsa_s": 0.12434525392018259,
  "cqsa_gib": 0.6370854377746582,
  "cqsa": "ok",
  "plan": "monolithic",
  "err_baseline": 0.00029992441232333446,
  "err_cqsa": 0.00029975092239689427
 },
 {
  "N": 262144,
  "baseline_s": 0.4295516440179199,
  "baseline_gib": 1.0159921646118164,
  